<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/Final_OT_Cybersecurity_Risk_Assessment_03b_January_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
from google.colab import userdata
from openai import OpenAI

# ---------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    if not os.path.exists(path):
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):

    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):

    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI 08-09_V6.pdf"

    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    R, R_expl = risk_estimator(L, I, heatmap_path)

    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    run_btn = gr.Button("Run Assessment")

    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.9/266.9 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.9/307.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: tomlkit
    Found existing installation: tomlkit 0.13.3
    Uninstalling tomlkit-0.13.

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    """
    Multi-factor likelihood agent:
    - Threat actor capability
    - Vulnerability exploitability (LLM)
    - Exposure
    - Historical occurrences (LLM)
    Final likelihood = max of the four factors.
    """
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    # Driving factor = highest severity
    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    # Explanation generation
    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://2282544a90e7c01730.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Modified to use public web APIs for Vulnerability Exploitability & Exposure)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (with Web Data)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD public API (no auth needed for low volume).
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org free API.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent (now web-informed):
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (LLM)
    Final likelihood = max of the four factors.
    """
    tac = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    # Defaults before enrichment
    vuln = None
    exp_factor = None

    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS
        epss = fetch_epss(cve)
        if epss is not None:
            exp_factor = map_epss_to_likelihood(epss)

    # Fallbacks if web data is not available
    if vuln is None:
        vuln = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_factor is None:
        exp_factor = clamp_likelihood(exposure_input)

    # Historical occurrences still via LLM
    hist = infer_likelihood_from_model(title + " (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp_factor],
        "historical_occurrences": L2S[hist]
    }

    # Driving factor = highest severity
    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    # Explanation generation
    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp_factor}, HIST={hist}.
Driving factor: {max_factor}.
If a CVE was available, mention how public exploitability and EPSS data influenced the assessment.
CVE used (if any): {cve}
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp_factor,
        "historical_occurrences": hist,
        "cve": cve
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (now web-informed for exploitability & exposure if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating
    R, R_expl = risk_estimator(L, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://fc9deb8f9c1fcc384c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + API-key integration)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# >>> CHANGED: Load external API keys from Colab userdata
# You should define these keys in Colab:
# userdata["nvd_api_key"], userdata["epss_api_key"], userdata["history_api_key"]
NVD_API_KEY = userdata.get("nvd_api_key")
EPSS_API_KEY = userdata.get("epss_api_key")          # optional; included per your request
HISTORY_API_KEY = userdata.get("history_api_key")    # for historical occurrence API

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Uses API key if available (if your plan requires it).
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    headers = {}
    params = {}

    # Some deployments might use API keys via headers or params.
    # Adjust according to your EPSS account requirements.
    if EPSS_API_KEY:
        headers["Authorization"] = f"Bearer {EPSS_API_KEY}"

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

# >>> CHANGED: Historical occurrence via external API

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    You can adapt URL/structure to your real service.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        # Expecting a float field "likelihood" between 0 and 1
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

# >>> CHANGED: New weighted + override likelihood_agent

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    # >>> CHANGED: likelihood_agent now returns (label, numeric, explanation, details)
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


SecretNotFoundError: Secret epss_api_key does not exist.

In [ ]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + API-key integration, free EPSS)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    You can adapt URL/structure to your real service.
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        # Expecting a float field "likelihood" between 0 and 1
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact.
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    """
    Multi-factor impact agent:
    - Safety, Availability, Confidentiality, Integrity (user inputs)
    - Compliance (RAG)
    - Reputation (LLM)
    Final impact = max of all six factors.
    """
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # Build RAG index for compliance PDF
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    # Explanation
    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://04ac62b20b84832287.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Weighted + RAG for Compliance)
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use ONLY the context below and your knowledge of regulatory principles:

{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations
- potential for regulatory findings or violations
- reporting obligations or enforcement actions

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Consider:
- public perception
- media attention
- stakeholder and investor confidence

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model:

    Factors (labels on 0–4 scale):
    - Safety
    - Availability
    - Confidentiality
    - Integrity
    - Compliance (RAG-based from FANR-REG-08 or similar)
    - Reputation (LLM-based)

    Weights (sum to 1.0, tuned for nuclear OT):
    - Safety:         0.35
    - Availability:   0.25
    - Integrity:      0.15
    - Confidentiality:0.10
    - Compliance:     0.10
    - Reputation:     0.05

    Conservative overrides:
    - If Safety >= major   → final impact >= major
    - If Compliance = severe → final impact >= major
    - If asset criticality = Very High → bump one level (capped at severe)
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
Highlight how the matrix embodies the organization's risk appetite and
the interaction between likelihood and impact categories.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library.
    """
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
Prioritize safety, defense-in-depth, and regulatory alignment for nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
Emphasize nuclear OT considerations, regulatory expectations, and
how the controls reduce likelihood and/or impact.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical JSON):
{json.dumps(likelihood_details, indent=2)}

Impact={impact}
Impact basis:
{impact_basis}

Impact details (technical JSON):
{json.dumps(impact_details, indent=2)}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
Include:
- Executive summary
- Risk description
- Likelihood analysis
- Impact analysis
- Overall risk rating
- Recommended controls and implementation priorities
- Notes for regulators and auditors
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations
    5. Final report generation
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://a4a321e683618821e5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [1]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# OT-NUCLEAR-FOCUSED PROMPTS AND CONTROLS
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# --------------------------------------------------
# Global OT-nuclear system prompt for all chat calls
# --------------------------------------------------

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain your reasoning in 1–2 short sentences in a nuclear OT context, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Explicitly reference OT realities such as legacy ICS/SCADA, limited patching windows,
  deterministic protocols, and physical process coupling where relevant.
Avoid generic IT-centric language.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Weighted + RAG for Compliance)
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model:

    Factors (labels on 0–4 scale):
    - Safety
    - Availability
    - Confidentiality
    - Integrity
    - Compliance (RAG-based from FANR-REG-08 or similar)
    - Reputation (LLM-based)

    Weights (sum to 1.0, tuned for nuclear OT):
    - Safety:         0.35
    - Availability:   0.25
    - Integrity:      0.15
    - Confidentiality:0.10
    - Compliance:     0.10
    - Reputation:     0.05

    Conservative overrides:
    - If Safety >= major        → final impact >= major
    - If Compliance = severe    → final impact >= major
    - If asset criticality = Very High → bump one level (capped at severe)
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation – OT-nuclear framing
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    # Strong OT-nuclear-focused RAG query
    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator (OT-nuclear framing)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a polished, structured OT nuclear cyber risk report.
    """
    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset category = {asset_category}
Asset criticality = {asset_criticality}
Risk title = {title}
Causes = {causes}

Likelihood = {likelihood_label} (numeric score={likelihood_numeric:.2f})
Likelihood basis:
{likelihood_basis}

Likelihood details (technical JSON):
{json.dumps(likelihood_details, indent=2)}

Impact = {impact}
Impact basis:
{impact_basis}

Impact details (technical JSON):
{json.dumps(impact_details, indent=2)}

Risk rating = {rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Controls rationale:
{controls_rationale}

Report requirements:
- Write professionally, as if for a nuclear OT cyber risk committee.
- Use headings and bullet points.
- Sections:
  1) Executive Summary (in nuclear OT terms, not generic IT)
  2) Risk Description (including asset, process, and safety relevance)
  3) Likelihood Analysis (highlight OT-specific drivers)
  4) Impact Analysis (safety, regulatory, operational, reputation)
  5) Overall Risk Rating (link to nuclear risk appetite)
  6) Recommended Controls and Implementation Priorities
  7) Notes for Regulators and Auditors (FANR/NEI/NRC alignment)

- Emphasize:
  • physical process and safety implications,
  • operational constraints (maintenance windows, legacy systems),
  • regulatory expectations,
  • why generic IT practices are insufficient without OT-specific adaptation.

Avoid cloud-centric or purely IT-enterprise language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations (OT-nuclear-focused)
    5. Final report generation (OT-nuclear framing)
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls (OT-nuclear-focused)
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report (OT-nuclear framing)
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n{I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n{R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full OT-nuclear-focused multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://de87453204e166600d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [3]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# (Web-enriched Likelihood + Weighted Impact + RAG Compliance)
# OT-NUCLEAR-FOCUSED PROMPTS AND CONTROLS
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------

# API key is securely stored in Colab's userdata store.
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Safe secret loading to avoid SecretNotFoundError
def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

# Optional external API keys
# Define these in Colab if you want authenticated access:
# userdata["nvd_api_key"], userdata["history_api_key"]
NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# Models used throughout the pipeline
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# --------------------------------------------------
# Global OT-nuclear system prompt for all chat calls
# --------------------------------------------------

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        return ""

    # Plain text
    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # CSV → convert to text
    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    # JSON → raw text
    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    # PDF → extract text page-by-page
    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    # DOCX → extract paragraphs
    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Weighted + Web + Historical)
# ============================================================

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain your reasoning in 1–2 short sentences in a nuclear OT context, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

# ---------------------------
# New helpers for web-driven data
# ---------------------------

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available.
    Returns exploitability score, impact score, and vector if available.
    """
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY  # NVD standard header

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        # Prefer CVSS v3.1 if present
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Free and open — no API key required.
    Returns a float in [0,1] if available.
    """
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom free API.
    Assumes an API that returns something like:
    {
      "cve": "CVE-2023-12345",
      "likelihood": 0.0-1.0  // probability based on historical events
    }
    If HISTORY_API_KEY or endpoint is not configured, returns None.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None

    # Placeholder URL – replace with your actual endpoint
    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS for consistency.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with:
    - Threat actor capability (user input)
    - Vulnerability exploitability (NVD CVSS exploitability if CVE available, else LLM)
    - Exposure (EPSS if CVE available, else user input)
    - Historical occurrences (external API if CVE + key, else LLM)

    Aggregation:
    - Compute weighted numeric score on a 0–4 scale
      TAC: 0.25, VULN: 0.35, EXP: 0.25, HIST: 0.15
    - Apply conservative override rules
    - Return both numeric score and final label, with explanation and details
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        # 1) Vulnerability exploitability via NVD
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        # 2) Exposure via EPSS (free, no key)
        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        # 3) Historical occurrence via external API (if configured)
        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Explicitly reference OT realities such as legacy ICS/SCADA, limited patching windows,
  deterministic protocols, and physical process coupling where relevant.
Avoid generic IT-centric language.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    # Return label + numeric score + explanation + structured details
    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Weighted + RAG for Compliance)
# ============================================================

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model:

    Factors (labels on 0–4 scale):
    - Safety
    - Availability
    - Confidentiality
    - Integrity
    - Compliance (RAG-based from FANR-REG-08 or similar)
    - Reputation (LLM-based)

    Weights (sum to 1.0, tuned for nuclear OT):
    - Safety:         0.35
    - Availability:   0.25
    - Integrity:      0.15
    - Confidentiality:0.10
    - Compliance:     0.10
    - Reputation:     0.05

    Conservative overrides:
    - If Safety >= major        → final impact >= major
    - If Compliance = severe    → final impact >= major
    - If asset criticality = Very High → bump one level (capped at severe)
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    # Explanation – OT-nuclear framing
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    # Strong OT-nuclear-focused RAG query
    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    # Rationale
    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator (OT-nuclear framing, structured Markdown)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact_label, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a structured OT nuclear cyber risk report in Markdown for Gradio.
    Uses the detailed likelihood/impact JSON to build a regulator-friendly,
    traceable report without additional model calls.
    """

    # Unpack likelihood details safely
    tac_label = likelihood_details.get("threat_actor_capability_label", "n/a")
    vuln_label = likelihood_details.get("vulnerability_exploitability_label", "n/a")
    exp_label = likelihood_details.get("exposure_label", "n/a")
    hist_label = likelihood_details.get("historical_occurrence_label", "n/a")

    tac_score = likelihood_details.get("threat_actor_capability_score", 0)
    vuln_score = likelihood_details.get("vulnerability_exploitability_score", 0)
    exp_score = likelihood_details.get("exposure_score", 0)
    hist_score = likelihood_details.get("historical_occurrence_score", 0)

    L_base_numeric = likelihood_details.get("base_numeric_score", likelihood_numeric)
    L_base_label = likelihood_details.get("base_label", likelihood_label)
    L_override_reason = likelihood_details.get("override_reason", "no override applied")
    cve_used = likelihood_details.get("cve") or "None detected"

    # Unpack impact details safely
    s_label = impact_details.get("safety_label", "n/a")
    a_label = impact_details.get("availability_label", "n/a")
    c_label = impact_details.get("confidentiality_label", "n/a")
    i_label = impact_details.get("integrity_label", "n/a")
    comp_label = impact_details.get("compliance_label", "n/a")
    rep_label = impact_details.get("reputation_label", "n/a")

    s_score = impact_details.get("safety_score", 0)
    a_score = impact_details.get("availability_score", 0)
    c_score = impact_details.get("confidentiality_score", 0)
    i_score = impact_details.get("integrity_score", 0)
    comp_score = impact_details.get("compliance_score", 0)
    rep_score = impact_details.get("reputation_score", 0)

    I_base_numeric = impact_details.get("base_numeric_score", 0)
    I_base_label = impact_details.get("base_label", impact_label)
    I_final_numeric = impact_details.get("final_numeric_score", 0)
    I_final_label = impact_details.get("final_label", impact_label)
    I_override_reason = impact_details.get("override_reason", "no override applied")

    weights = impact_details.get("weights", {})
    w_safety = weights.get("safety", 0.35)
    w_availability = weights.get("availability", 0.25)
    w_integrity = weights.get("integrity", 0.15)
    w_confidentiality = weights.get("confidentiality", 0.10)
    w_compliance = weights.get("compliance", 0.10)
    w_reputation = weights.get("reputation", 0.05)

    # Executive summary text (simple, deterministic)
    exec_summary = (
        f"The assessed risk '{title}' affects an '{asset_category}' asset with '{asset_criticality}' criticality in the "
        f"nuclear OT environment. The multi-factor likelihood model, informed by threat actor capability, "
        f"vulnerability exploitability, exposure, and historical occurrence, results in a final likelihood of "
        f"**{likelihood_label}** (score={likelihood_numeric:.2f}). The impact model, which prioritizes safety, "
        f"availability, and regulatory compliance, results in a final impact of **{impact_label}** "
        f"(score={I_final_numeric:.2f}). Combined on the nuclear OT risk heatmap, this yields an overall "
        f"risk rating of **{rating}**. Conservative overrides are applied where safety, compliance, or very high "
        f"asset criticality warrant elevation of the result in line with nuclear safety culture and regulatory "
        f"expectations."
    )

    report = f"""
# OT Nuclear Cyber Risk Report

## 1) Executive Summary

{exec_summary}

---

## 2) Risk Description

- **Risk Title:** {title}
- **Asset Category:** {asset_category}
- **Asset Criticality:** {asset_criticality}
- **Primary Causes / Scenario:** {causes}
- **Detected CVE (if any):** {cve_used}

This risk is evaluated in the context of nuclear operational technology, where deterministic control system behavior,
physical process coupling, and limited maintenance windows require conservative assumptions and defense-in-depth
across safety-related and important-to-safety assets.

---

## 3) Likelihood Analysis

**Final Likelihood:** **{likelihood_label}** (score={likelihood_numeric:.2f})
**Base Likelihood (before overrides):** {L_base_label} (score={L_base_numeric:.2f})
**Overrides Applied:** {L_override_reason}

### 3.1 Factor Breakdown

- **Threat Actor Capability:** {tac_label} (score={tac_score})
- **Vulnerability Exploitability:** {vuln_label} (score={vuln_score})
- **Exposure:** {exp_label} (score={exp_score})
- **Historical Occurrence:** {hist_label} (score={hist_score})

These factors are combined using a weighted 0–4 scale tailored to nuclear OT, with higher emphasis on vulnerability
exploitability and realistic exposure conditions, while maintaining a conservative floor when both are elevated.

<details>
<summary>Detailed Likelihood Explanation</summary>

{likelihood_basis}

</details>

---

## 4) Impact Analysis

**Final Impact:** **{I_final_label}** (score={I_final_numeric:.2f})
**Base Impact (before overrides):** {I_base_label} (score={I_base_numeric:.2f})
**Overrides Applied:** {I_override_reason}

### 4.1 Factor Breakdown (0–4 scale)

- **Safety:** {s_label} (score={s_score})
- **Availability:** {a_label} (score={a_score})
- **Confidentiality:** {c_label} (score={c_score})
- **Integrity:** {i_label} (score={i_score})
- **Compliance / Regulatory:** {comp_label} (score={comp_score})
- **Reputation / Public Confidence:** {rep_label} (score={rep_score})

### 4.2 Weighting Emphasis

- **Safety weight:** {w_safety}
- **Availability weight:** {w_availability}
- **Integrity weight:** {w_integrity}
- **Confidentiality weight:** {w_confidentiality}
- **Compliance weight:** {w_compliance}
- **Reputation weight:** {w_reputation}

Safety and regulatory consequences are deliberately given the highest influence, reflecting the priority of preventing
adverse effects on nuclear safety functions, plant availability for safe operation, and compliance with FANR/NEI/NRC
requirements.

<details>
<summary>Detailed Impact Explanation</summary>

{impact_basis}

</details>

---

## 5) Overall Risk Rating

- **Risk Rating (Heatmap Result):** **{rating}**
- **Likelihood (final):** {likelihood_label} (score={likelihood_numeric:.2f})
- **Impact (final):** {I_final_label} (score={I_final_numeric:.2f})

The rating is derived from a nuclear OT-specific likelihood × impact matrix that encodes the organization's
risk appetite for safety-related and important-to-safety systems.

<details>
<summary>Heatmap Rating Explanation</summary>

{rating_expl}

</details>

---

## 6) Recommended Controls and Implementation Priorities

Below controls are selected with preference for engineering, procedural, and physical safeguards aligned to NEI 08-09,
FANR-REG-08, NRC RG 5.71, and ISA/IEC 62443 as adapted for nuclear facilities.

{controls}

<details>
<summary>Control Rationale</summary>

{controls_rationale}

</details>

---

## 7) Notes for Regulators and Auditors

- The likelihood model incorporates structured inputs for threat actor capability, exploitability, exposure, and
  historical occurrence, with explicit conservative override rules documented above.
- The impact model reflects nuclear safety culture by giving priority to safety and compliance, and by elevating
  results for very high criticality assets where required.
- Controls are derived from a nuclear OT-focused control library using retrieval-augmented generation, ensuring
  traceability back to NEI 08-09 and similar frameworks.
- Generic enterprise IT practices are only adopted where explicitly compatible with deterministic OT behavior,
  maintenance constraints, and segregation between safety, non-safety, and corporate networks.
"""

    return report

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations (OT-nuclear-focused)
    5. Final report generation (OT-nuclear framing)
    """
    compliance_path = "data/FANR-REG-08_V2.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        compliance_path,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, heatmap_path)

    # Step 4: Controls (OT-nuclear-focused)
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # Step 5: Final Report (OT-nuclear framing, structured Markdown)
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    # For UI, show both label and numeric in the Likelihood section
    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n**Final Impact:** {I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full OT-nuclear-focused multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://b5755f341989fbcab1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [4]:
# ============================================================
# OT Nuclear Risk Assessment – Optimized Version
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2 requests

import os
import io
import json
import re
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
import requests
from functools import lru_cache
from google.colab import userdata
from openai import OpenAI

# ============================================================
# 0) CONFIG & FLAGS
# ============================================================

# Models
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# Data paths (adjust if needed)
COMPLIANCE_PATH = "data/FANR-REG-08_V2.pdf"
HEATMAP_PATH = "data/Nuclear_OT_Risk_Heatmap.xlsx"
CONTROL_PATH = "data/NEI_08_09_FULL_Cybersecurity_Control_Library.xlsx"

# Performance / behavior flags
DEBUG_LOG = False   # If True, prints key timing/info to console (minimal)
TEST_MODE = False   # If True, you can mock / simplify external behavior

# Likelihood scale
LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

# Impact scale
IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

# --------------------------------------------------
# 1) Configure OpenAI client using Colab Secrets
# --------------------------------------------------

os.environ["OPENAI_API_KEY"] = userdata.get("openai")

def safe_get_secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

NVD_API_KEY = safe_get_secret("nvd_api_key")
HISTORY_API_KEY = safe_get_secret("history_api_key")

if NVD_API_KEY is None:
    print("⚠️ NVD_API_KEY not found — using unauthenticated NVD mode (rate-limited).")

if HISTORY_API_KEY is None:
    print("⚠️ HISTORY_API_KEY not found — historical occurrence API disabled (LLM fallback).")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

# --------------------------------------------------
# Global OT-nuclear system prompt
# --------------------------------------------------

NUCLEAR_OT_SYSTEM_PROMPT = """
You are an expert in nuclear Operational Technology (OT) cybersecurity.

Your answers MUST be grounded in:
- NEI 08-09 (Cyber Security Plan for Nuclear Power Reactors),
- NRC RG 5.71,
- FANR-REG-08,
- ISA/IEC 62443 as adapted for nuclear facilities,
- safety-critical engineering principles for nuclear plants.

STRICT REQUIREMENTS:
- Focus on OT systems (control systems, safety systems, engineering workstations, process networks).
- Emphasize deterministic system behavior, safety-first design, and defense-in-depth for physical processes.
- Account for legacy systems, limited patching windows, maintenance constraints, and physical process coupling.
- Align with regulatory expectations and nuclear safety culture.
- Avoid generic enterprise IT controls and tools (e.g., SIEM, EDR, CASB, DLP, cloud-first architectures) unless explicitly and clearly justified as applicable to nuclear OT.
- Do NOT recommend cloud-based solutions for critical OT functions.
- Prefer engineering, procedural, and physical controls over purely IT-centric technical controls.
"""

# ============================================================
# 2) Utility: Embeddings + RAG (Optimized with caching)
# ============================================================

EMBED_CACHE = {}

def get_embedding(text: str):
    """
    Generate an embedding vector for a given text.
    Newlines removed to avoid embedding inconsistencies.
    """
    text = text.replace("\n", " ")
    if TEST_MODE:
        # Deterministic simple embedding for test mode
        return [hash(text) % 997 / 997.0] * 16

    # Use cache to avoid recomputing embeddings
    key = ("emb", EMBEDDING_MODEL, text)
    if key in EMBED_CACHE:
        return EMBED_CACHE[key]
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    emb = resp.data[0].embedding
    EMBED_CACHE[key] = emb
    return emb

def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """
    Read text from multiple file formats:
    txt, csv, json, pdf, docx.
    Returns raw text for RAG indexing.
    """
    if not os.path.exists(path):
        if DEBUG_LOG:
            print(f"[DEBUG] File not found for RAG: {path}")
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

@lru_cache(maxsize=16)
def build_rag_index(path: str):
    """
    Build a simple RAG index:
    - Load file text
    - Chunk into ~700-word segments
    - Embed each chunk

    Cached via lru_cache so built once per path.
    """
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    if DEBUG_LOG:
        print(f"[DEBUG] RAG index built for {path}, chunks={len(index)}")
    return index

def rag_search(index, query, top_k=3):
    """
    Retrieve top-k most relevant chunks using cosine similarity.
    """
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent (Optimized)
# ============================================================

def clamp_likelihood(v):
    """Normalize model output to one of the allowed likelihood labels."""
    if v is None:
        return "possible"
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    """
    Ask the LLM to classify likelihood based on title + causes.
    Used as a fallback when web data is unavailable.
    """
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Explain your reasoning in 1–2 short sentences in a nuclear OT context, then output ONLY the label on the last line.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content.splitlines()[-1])

def extract_cve(text):
    """
    Extract the first CVE identifier if present in the text.
    Example: CVE-2023-12345
    """
    match = re.search(r"CVE-\d{4}-\d{4,7}", text, flags=re.IGNORECASE)
    return match.group(0).upper() if match else None

@lru_cache(maxsize=512)
def fetch_nvd_data(cve_id):
    """
    Fetch vulnerability data from NVD API.
    Uses API key if available. Cached per CVE.
    """
    if TEST_MODE:
        return {"exploitability": 2.5, "impact": 3.0, "vector": "TEST/AV:N/..."}
    url = f"https://services.nvd.nist.gov/rest/json/cve/2.0?cveId={cve_id}"
    headers = {}
    if NVD_API_KEY:
        headers["apiKey"] = NVD_API_KEY

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return None
        cve = vulns[0].get("cve", {})
        metrics = cve.get("metrics", {})
        cvss_list = metrics.get("cvssMetricV31") or metrics.get("cvssMetricV30") or metrics.get("cvssMetricV2")
        if not cvss_list:
            return None
        m = cvss_list[0]
        exploitability = m.get("exploitabilityScore")
        impact = m.get("impactScore")
        vector = m.get("cvssData", {}).get("vectorString")
        return {
            "exploitability": exploitability,
            "impact": impact,
            "vector": vector
        }
    except Exception:
        return None

def map_exploitability_to_likelihood(score):
    """
    Map CVSS exploitability score (0.0–3.9) to qualitative likelihood.
    """
    if score is None:
        return None
    try:
        s = float(score)
    except ValueError:
        return None

    if s <= 0.5:
        return "very unlikely"
    if s <= 1.5:
        return "unlikely"
    if s <= 2.5:
        return "possible"
    if s <= 3.2:
        return "likely"
    return "very likely"

@lru_cache(maxsize=512)
def fetch_epss(cve_id):
    """
    Fetch EPSS (Exploit Prediction Scoring System) probability from FIRST.org API.
    Cached per CVE.
    """
    if TEST_MODE:
        return 0.35
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        items = data.get("data", [])
        if not items:
            return None
        epss_str = items[0].get("epss")
        if epss_str is None:
            return None
        return float(epss_str)
    except Exception:
        return None

def map_epss_to_likelihood(epss):
    """
    Map EPSS probability (0–1) to qualitative likelihood for 'exposure'.
    """
    if epss is None:
        return None
    if epss < 0.05:
        return "very unlikely"
    if epss < 0.20:
        return "unlikely"
    if epss < 0.50:
        return "possible"
    if epss < 0.75:
        return "likely"
    return "very likely"

@lru_cache(maxsize=512)
def fetch_historical_occurrence(cve_id):
    """
    Fetch historical occurrence data from a custom API.
    Cached per CVE.
    """
    if not HISTORY_API_KEY or not cve_id:
        return None
    if TEST_MODE:
        return 0.25

    url = "https://your-history-api.example.com/v1/history"
    headers = {"Authorization": f"Bearer {HISTORY_API_KEY}"}
    params = {"cve": cve_id}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        p = data.get("likelihood")
        if p is None:
            return None
        p = float(p)
        p = max(0.0, min(1.0, p))
        return p
    except Exception:
        return None

def map_history_prob_to_likelihood(p):
    """
    Map historical probability (0–1) to qualitative likelihood.
    Similar breakpoints to EPSS.
    """
    if p is None:
        return None
    if p < 0.05:
        return "very unlikely"
    if p < 0.20:
        return "unlikely"
    if p < 0.50:
        return "possible"
    if p < 0.75:
        return "likely"
    return "very likely"

def likelihood_agent(threat_actor, exposure_input, title, causes):
    """
    Multi-factor likelihood agent with caching and minimized external calls.
    """

    # Normalize TAC from user
    tac_label = clamp_likelihood(threat_actor)

    # Combine title + causes to search for CVE IDs
    cve = extract_cve(f"{title} {causes}")

    vuln_label = None
    exp_label = None
    hist_label = None

    # Web-enriched paths if CVE is present
    if cve:
        nvd = fetch_nvd_data(cve)
        if nvd and nvd.get("exploitability") is not None:
            vuln_label = map_exploitability_to_likelihood(nvd["exploitability"])

        epss = fetch_epss(cve)
        if epss is not None:
            exp_label = map_epss_to_likelihood(epss)

        hist_prob = fetch_historical_occurrence(cve)
        if hist_prob is not None:
            hist_label = map_history_prob_to_likelihood(hist_prob)

    # Fallbacks if web data is not available
    if vuln_label is None:
        vuln_label = infer_likelihood_from_model(title + " (vuln)", causes)

    if exp_label is None:
        exp_label = clamp_likelihood(exposure_input)

    if hist_label is None:
        hist_label = infer_likelihood_from_model(title + " (history)", causes)

    # Convert to numeric scores 0–4
    tac_score = L2S[tac_label]
    vuln_score = L2S[vuln_label]
    exp_score = L2S[exp_label]
    hist_score = L2S[hist_label]

    # Weighted aggregation
    w_tac = 0.25
    w_vuln = 0.35
    w_exp = 0.25
    w_hist = 0.15

    base_numeric = (
        w_tac * tac_score
        + w_vuln * vuln_score
        + w_exp * exp_score
        + w_hist * hist_score
    )

    # Round to nearest integer for base label
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2L[base_index]

    # Conservative override rules
    final_numeric = base_numeric
    override_reason = None

    # Rule 1: If VULN >= likely AND EXP >= likely ⇒ floor "likely"
    if vuln_score >= L2S["likely"] and exp_score >= L2S["likely"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            override_reason = (
                "Raised to at least 'likely' because both vulnerability exploitability "
                "and exposure are high."
            )

    # Rule 2: Emerging threat – very high exploitability, non-trivial exposure
    if vuln_score >= L2S["very likely"] and exp_score >= L2S["possible"]:
        if final_numeric < L2S["likely"]:
            final_numeric = float(L2S["likely"])
            if override_reason:
                override_reason += " Additionally, exploitability is very high with non-trivial exposure."
            else:
                override_reason = (
                    "Raised to at least 'likely' because exploitability is very high "
                    "and exposure is at least possible."
                )

    # Clamp and map final score
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2L[final_index]

    # Explanation generation (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT likelihood assessment.

Scales:
0=very unlikely, 1=unlikely, 2=possible, 3=likely, 4=very likely.

Inputs (labels and scores):
- Threat Actor Capability: {tac_label} (score={tac_score})
- Vulnerability Exploitability: {vuln_label} (score={vuln_score})
- Exposure: {exp_label} (score={exp_score})
- Historical Occurrence: {hist_label} (score={hist_score})

Weighted model:
- TAC weight = {w_tac}
- VULN weight = {w_vuln}
- EXP weight = {w_exp}
- HIST weight = {w_hist}

Base numeric likelihood = {base_numeric:.2f}, base label = {base_label}.
Final numeric likelihood = {final_numeric:.2f}, final label = {final_label}.

Override reason (if any): {override_reason or "no override applied"}.
CVE used (if any): {cve}

Explain in 4–7 sentences:
- How the weighted model arrived at the base score,
- Why any overrides were applied or not,
- Why the final label is appropriate for a nuclear OT context,
- Explicitly reference OT realities such as legacy ICS/SCADA, limited patching windows,
  deterministic protocols, and physical process coupling where relevant.
Avoid generic IT-centric language.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.5
    )

    details = {
        "threat_actor_capability_label": tac_label,
        "vulnerability_exploitability_label": vuln_label,
        "exposure_label": exp_label,
        "historical_occurrence_label": hist_label,
        "threat_actor_capability_score": tac_score,
        "vulnerability_exploitability_score": vuln_score,
        "exposure_score": exp_score,
        "historical_occurrence_score": hist_score,
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
        "cve": cve
    }

    return final_label, final_numeric, exp_resp.choices[0].message.content, details

# ============================================================
# 4) Impact Calculator Agent (Optimized + RAG caching)
# ============================================================

def clamp_impact(v):
    """Normalize model output to valid impact label."""
    if v is None:
        return "moderate"
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    """
    Use RAG context to infer compliance impact from FANR-REG-08 (or other compliance document).
    """
    ctx = rag_search(
        rag_index,
        f"Compliance impact for nuclear OT cyber risk: {title}. Causes: {causes}. "
        f"Focus on licensing, regulatory obligations, reportable events, and enforcement actions.",
        3
    )
    prompt = f"""
You are assessing compliance impact for a nuclear OT cyber risk.

Use the context below from regulatory documents (e.g., FANR-REG-08, NEI 08-09, NRC RG 5.71) as your
PRIMARY source of truth. If the context is weak, apply conservative nuclear regulatory judgment.

Context:
{ctx}

Map compliance impact to one of:
negligible, marginal, moderate, major, severe.

Focus on:
- licensing and regulatory obligations,
- potential for regulatory findings, violations, or enforcement actions,
- reportable events and escalation paths.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def infer_reputation(asset_category, title, causes):
    """
    LLM-based reputation impact classifier.
    """
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Context:
- Asset: {asset_category}
- Risk: {title}
- Causes: {causes}

Consider:
- public perception and media attention specific to nuclear facilities,
- stakeholder, regulator, and investor confidence,
- potential societal concern related to nuclear safety and security.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content.strip().splitlines()[-1])

def impact_agent(asset_category,
                 asset_criticality,
                 safety,
                 availability,
                 confidentiality,
                 integrity,
                 compliance_path,
                 title,
                 causes):
    """
    Multi-factor impact agent using a weighted model, with RAG index cached.
    """

    # 1) Normalize user inputs
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    # 2) RAG index for compliance document and infer compliance impact
    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)

    # 3) LLM-based reputation impact
    rep = infer_reputation(asset_category, title, causes)

    # 4) Convert to numeric scores (0–4)
    safety_score = I2S[s]
    availability_score = I2S[a]
    confidentiality_score = I2S[c]
    integrity_score = I2S[i]
    compliance_score = I2S[comp]
    reputation_score = I2S[rep]

    # 5) Weighted aggregation
    w_safety = 0.35
    w_availability = 0.25
    w_integrity = 0.15
    w_confidentiality = 0.10
    w_compliance = 0.10
    w_reputation = 0.05

    base_numeric = (
        w_safety * safety_score
        + w_availability * availability_score
        + w_integrity * integrity_score
        + w_confidentiality * confidentiality_score
        + w_compliance * compliance_score
        + w_reputation * reputation_score
    )

    # Map numeric to base label (0–4 → negligible–severe)
    base_index = int(round(base_numeric))
    base_index = max(0, min(4, base_index))
    base_label = S2I[base_index]

    # 6) Conservative override rules
    final_numeric = base_numeric
    override_reasons = []

    # Rule 1: Safety dominates for high-severity events
    if safety_score >= I2S["major"]:
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' because safety impact is high."
            )

    # Rule 2: Severe compliance impact (e.g., major regulatory violation)
    if comp == "severe":
        if final_numeric < I2S["major"]:
            final_numeric = float(I2S["major"])
            override_reasons.append(
                "Raised to at least 'major' due to severe compliance impact."
            )

    # Rule 3: Very High criticality assets → bump one level
    if asset_criticality and asset_criticality.lower() == "very high":
        bumped = min(4, int(round(final_numeric)) + 1)
        if bumped > int(round(final_numeric)):
            final_numeric = float(bumped)
            override_reasons.append(
                "Impact bumped one level because asset criticality is Very High."
            )

    # Clamp again and map to final label
    final_index = int(round(final_numeric))
    final_index = max(0, min(4, final_index))
    final_label = S2I[final_index]

    override_reason = "; ".join(override_reasons) if override_reasons else "no override applied"

    # 7) Explanation via LLM (OT-nuclear framing)
    explanation_prompt = f"""
You are explaining a nuclear OT impact assessment.

Impact scale:
0=negligible, 1=marginal, 2=moderate, 3=major, 4=severe.

Inputs (labels and scores):
- Safety:          {s} (score={safety_score})
- Availability:    {a} (score={availability_score})
- Confidentiality: {c} (score={confidentiality_score})
- Integrity:       {i} (score={integrity_score})
- Compliance:      {comp} (score={compliance_score})
- Reputation:      {rep} (score={reputation_score})

Weights:
- Safety weight         = {w_safety}
- Availability weight   = {w_availability}
- Integrity weight      = {w_integrity}
- Confidentiality weight= {w_confidentiality}
- Compliance weight     = {w_compliance}
- Reputation weight     = {w_reputation}

Asset criticality: {asset_criticality}

Base numeric impact = {base_numeric:.2f}, base label = {base_label}.
Final numeric impact = {final_numeric:.2f}, final label = {final_label}.

Override reasons: {override_reason}.

Explain in 4–7 sentences:
- How the weighted model produced the base score,
- Why any overrides were applied or not applied,
- Why the final impact label is appropriate for a nuclear OT context,
- How safety, compliance, and asset criticality influenced conservatism,
- Explicitly reference physical process consequences and nuclear safety where relevant.
Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": explanation_prompt}
        ],
        temperature=0.4
    )

    impact_details = {
        "safety_label": s,
        "availability_label": a,
        "confidentiality_label": c,
        "integrity_label": i,
        "compliance_label": comp,
        "reputation_label": rep,
        "safety_score": safety_score,
        "availability_score": availability_score,
        "confidentiality_score": confidentiality_score,
        "integrity_score": integrity_score,
        "compliance_score": compliance_score,
        "reputation_score": reputation_score,
        "weights": {
            "safety": w_safety,
            "availability": w_availability,
            "integrity": w_integrity,
            "confidentiality": w_confidentiality,
            "compliance": w_compliance,
            "reputation": w_reputation,
        },
        "base_numeric_score": base_numeric,
        "base_label": base_label,
        "final_numeric_score": final_numeric,
        "final_label": final_label,
        "override_reason": override_reason,
    }

    return final_label, resp.choices[0].message.content, impact_details

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

@lru_cache(maxsize=4)
def load_heatmap_df(path):
    df = pd.read_excel(path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])
    return df

def risk_estimator(likelihood, impact, heatmap_path):
    """
    Look up risk rating from a likelihood × impact heatmap (Excel).
    """
    df = load_heatmap_df(heatmap_path)

    like = likelihood.lower()
    imp = impact.lower()

    if like not in df.index:
        raise ValueError(f"Likelihood '{likelihood}' not found in heatmap.")
    if imp not in df.columns:
        raise ValueError(f"Impact '{impact}' not found in heatmap columns.")

    rating = df.loc[like, imp]

    # Explanation – OT-nuclear framing
    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.

Focus on:
- how the matrix reflects nuclear safety culture and risk appetite,
- how higher impact categories reflect potential physical consequences and regulatory concern,
- how likelihood interacts with impact for operational technology, not generic IT.

Avoid generic IT-centric language.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG, OT-nuclear-focused)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    """
    Recommend controls using RAG from NEI 08-09 control library,
    strongly constrained to OT-nuclear context.
    """
    index = build_rag_index(control_path)

    query = (
        f"Nuclear OT cybersecurity controls for asset type '{asset_category}' with criticality '{asset_criticality}'. "
        f"Risk title: {title}. Causes: {causes}. Risk rating: {risk_rating}. "
        f"Focus on NEI 08-09, defense-in-depth, safety systems, engineering workstations, "
        f"segmentation of safety and non-safety systems, and physical process protection."
    )
    ctx = rag_search(index, query, 5)

    prompt = f"""
You are selecting controls from an OT-nuclear-focused control library (e.g., NEI 08-09).

Control library context (PRIMARY source of truth):
{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}
Asset criticality: {asset_criticality}
Risk title: {title}
Causes: {causes}

Provide OT-nuclear-specific controls ONLY.

STRICT REQUIREMENTS:
- Use the control library context as the PRIMARY basis; general knowledge is secondary.
- Controls MUST align with nuclear OT principles:
  • deterministic system behavior,
  • safety-first design and protection of safety-related systems,
  • defense-in-depth for physical processes,
  • regulatory alignment (NEI 08-09, FANR-REG-08, NRC RG 5.71),
  • engineering constraints and maintenance windows,
  • network isolation and segmentation between safety/non-safety and OT/IT.
- Prefer engineering, procedural, and physical controls over generic IT controls.
- Avoid recommending enterprise IT tools (SIEM, EDR, CASB, DLP, cloud-based monitoring) unless
  they are explicitly relevant and clearly justified for nuclear OT.
- Do NOT suggest cloud-based solutions for critical OT functions.

Output:
- A short heading for the control strategy.
- Bulleted list of recommended controls.
- A brief note for each control explaining how it reduces likelihood and/or impact in nuclear OT.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}' in a nuclear OT context.

Emphasize:
- nuclear safety culture and regulatory expectations,
- how the controls protect physical processes and safety systems,
- how they align with NEI 08-09 / FANR-REG-08-like control objectives,
- why they are preferable to generic IT cybersecurity practices for this scenario.

Avoid generic IT-centric phrasing.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": NUCLEAR_OT_SYSTEM_PROMPT},
            {"role": "user", "content": rationale_prompt}
        ],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator (OT-nuclear framing, structured Markdown)
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood_label, likelihood_numeric, likelihood_basis, likelihood_details,
                    impact_label, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):
    """
    Generate a structured OT nuclear cyber risk report in Markdown for Gradio.
    """

    # Unpack likelihood details safely
    tac_label = likelihood_details.get("threat_actor_capability_label", "n/a")
    vuln_label = likelihood_details.get("vulnerability_exploitability_label", "n/a")
    exp_label = likelihood_details.get("exposure_label", "n/a")
    hist_label = likelihood_details.get("historical_occurrence_label", "n/a")

    tac_score = likelihood_details.get("threat_actor_capability_score", 0)
    vuln_score = likelihood_details.get("vulnerability_exploitability_score", 0)
    exp_score = likelihood_details.get("exposure_score", 0)
    hist_score = likelihood_details.get("historical_occurrence_score", 0)

    L_base_numeric = likelihood_details.get("base_numeric_score", likelihood_numeric)
    L_base_label = likelihood_details.get("base_label", likelihood_label)
    L_override_reason = likelihood_details.get("override_reason", "no override applied")
    cve_used = likelihood_details.get("cve") or "None detected"

    # Unpack impact details safely
    s_label = impact_details.get("safety_label", "n/a")
    a_label = impact_details.get("availability_label", "n/a")
    c_label = impact_details.get("confidentiality_label", "n/a")
    i_label = impact_details.get("integrity_label", "n/a")
    comp_label = impact_details.get("compliance_label", "n/a")
    rep_label = impact_details.get("reputation_label", "n/a")

    s_score = impact_details.get("safety_score", 0)
    a_score = impact_details.get("availability_score", 0)
    c_score = impact_details.get("confidentiality_score", 0)
    i_score = impact_details.get("integrity_score", 0)
    comp_score = impact_details.get("compliance_score", 0)
    rep_score = impact_details.get("reputation_score", 0)

    I_base_numeric = impact_details.get("base_numeric_score", 0)
    I_base_label = impact_details.get("base_label", impact_label)
    I_final_numeric = impact_details.get("final_numeric_score", 0)
    I_final_label = impact_details.get("final_label", impact_label)
    I_override_reason = impact_details.get("override_reason", "no override applied")

    weights = impact_details.get("weights", {})
    w_safety = weights.get("safety", 0.35)
    w_availability = weights.get("availability", 0.25)
    w_integrity = weights.get("integrity", 0.15)
    w_confidentiality = weights.get("confidentiality", 0.10)
    w_compliance = weights.get("compliance", 0.10)
    w_reputation = weights.get("reputation", 0.05)

    exec_summary = (
        f"The assessed risk '{title}' affects an '{asset_category}' asset with '{asset_criticality}' criticality in the "
        f"nuclear OT environment. The multi-factor likelihood model, informed by threat actor capability, "
        f"vulnerability exploitability, exposure, and historical occurrence, results in a final likelihood of "
        f"**{likelihood_label}** (score={likelihood_numeric:.2f}). The impact model, which prioritizes safety, "
        f"availability, and regulatory compliance, results in a final impact of **{impact_label}** "
        f"(score={I_final_numeric:.2f}). Combined on the nuclear OT risk heatmap, this yields an overall "
        f"risk rating of **{rating}**. Conservative overrides are applied where safety, compliance, or very high "
        f"asset criticality warrant elevation of the result in line with nuclear safety culture and regulatory "
        f"expectations."
    )

    report = f"""
# OT Nuclear Cyber Risk Report

## 1) Executive Summary

{exec_summary}

---

## 2) Risk Description

- **Risk Title:** {title}
- **Asset Category:** {asset_category}
- **Asset Criticality:** {asset_criticality}
- **Primary Causes / Scenario:** {causes}
- **Detected CVE (if any):** {cve_used}

This risk is evaluated in the context of nuclear operational technology, where deterministic control system behavior,
physical process coupling, and limited maintenance windows require conservative assumptions and defense-in-depth
across safety-related and important-to-safety assets.

---

## 3) Likelihood Analysis

**Final Likelihood:** **{likelihood_label}** (score={likelihood_numeric:.2f})
**Base Likelihood (before overrides):** {L_base_label} (score={L_base_numeric:.2f})
**Overrides Applied:** {L_override_reason}

### 3.1 Factor Breakdown

- **Threat Actor Capability:** {tac_label} (score={tac_score})
- **Vulnerability Exploitability:** {vuln_label} (score={vuln_score})
- **Exposure:** {exp_label} (score={exp_score})
- **Historical Occurrence:** {hist_label} (score={hist_score})

These factors are combined using a weighted 0–4 scale tailored to nuclear OT, with higher emphasis on vulnerability
exploitability and realistic exposure conditions, while maintaining a conservative floor when both are elevated.

<details>
<summary>Detailed Likelihood Explanation</summary>

{likelihood_basis}

</details>

---

## 4) Impact Analysis

**Final Impact:** **{I_final_label}** (score={I_final_numeric:.2f})
**Base Impact (before overrides):** {I_base_label} (score={I_base_numeric:.2f})
**Overrides Applied:** {I_override_reason}

### 4.1 Factor Breakdown (0–4 scale)

- **Safety:** {s_label} (score={s_score})
- **Availability:** {a_label} (score={a_score})
- **Confidentiality:** {c_label} (score={c_score})
- **Integrity:** {i_label} (score={i_score})
- **Compliance / Regulatory:** {comp_label} (score={comp_score})
- **Reputation / Public Confidence:** {rep_label} (score={rep_score})

### 4.2 Weighting Emphasis

- **Safety weight:** {w_safety}
- **Availability weight:** {w_availability}
- **Integrity weight:** {w_integrity}
- **Confidentiality weight:** {w_confidentiality}
- **Compliance weight:** {w_compliance}
- **Reputation weight:** {w_reputation}

Safety and regulatory consequences are deliberately given the highest influence, reflecting the priority of preventing
adverse effects on nuclear safety functions, plant availability for safe operation, and compliance with FANR/NEI/NRC
requirements.

<details>
<summary>Detailed Impact Explanation</summary>

{impact_basis}

</details>

---

## 5) Overall Risk Rating

- **Risk Rating (Heatmap Result):** **{rating}**
- **Likelihood (final):** {likelihood_label} (score={likelihood_numeric:.2f})
- **Impact (final):** {I_final_label} (score={I_final_numeric:.2f})

The rating is derived from a nuclear OT-specific likelihood × impact matrix that encodes the organization's
risk appetite for safety-related and important-to-safety systems.

<details>
<summary>Heatmap Rating Explanation</summary>

{rating_expl}

</details>

---

## 6) Recommended Controls and Implementation Priorities

Below controls are selected with preference for engineering, procedural, and physical safeguards aligned to NEI 08-09,
FANR-REG-08, NRC RG 5.71, and ISA/IEC 62443 as adapted for nuclear facilities.

{controls}

<details>
<summary>Control Rationale</summary>

{controls_rationale}

</details>

---

## 7) Notes for Regulators and Auditors

- The likelihood model incorporates structured inputs for threat actor capability, exploitability, exposure, and
  historical occurrence, with explicit conservative override rules documented above.
- The impact model reflects nuclear safety culture by giving priority to safety and compliance, and by elevating
  results for very high criticality assets where required.
- Controls are derived from a nuclear OT-focused control library using retrieval-augmented generation, ensuring
  traceability back to NEI 08-09 and similar frameworks.
- Generic enterprise IT practices are only adopted where explicitly compatible with deterministic OT behavior,
  maintenance constraints, and segregation between safety, non-safety, and corporate networks.
"""

    return report

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING (Optimized)
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):
    """
    Orchestrates the full multi-agent pipeline:
    1. Likelihood agent (weighted + overrides, web-informed for exploitability, exposure, and history if CVE present)
    2. Impact agent (weighted model + RAG-based compliance impact)
    3. Heatmap risk rating
    4. Control recommendations (OT-nuclear-focused)
    5. Final report generation (OT-nuclear framing)
    """

    # Step 1: Likelihood
    L_label, L_numeric, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # Step 2: Impact (weighted + RAG compliance)
    I_label, I_basis, I_details = impact_agent(
        asset_category,
        asset_criticality,
        safety,
        availability,
        confidentiality,
        integrity,
        COMPLIANCE_PATH,
        risk_title,
        risk_causes
    )

    # Step 3: Risk Rating (uses label, not numeric)
    R, R_expl = risk_estimator(L_label, I_label, HEATMAP_PATH)

    # Step 4: Controls (OT-nuclear-focused)
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, CONTROL_PATH
    )

    # Step 5: Final Report (OT-nuclear framing, structured Markdown)
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L_label, L_numeric, L_basis, L_details,
        I_label, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    likelihood_md = (
        f"### Likelihood\n"
        f"**Final Likelihood:** {L_label} (score={L_numeric:.2f})\n\n"
        f"{L_basis}"
    )

    impact_md = f"### Impact\n**Final Impact:** {I_label}\n\n{I_basis}"
    risk_md = f"### Risk Rating\n**Rating:** {R}\n\n{R_expl}"
    controls_md = f"### Controls\n{C}\n\n### Rationale\n{C_rat}"

    return (
        likelihood_md,
        impact_md,
        risk_md,
        controls_md,
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full OT-nuclear-focused multi‑agent risk assessment.")

    # Asset metadata
    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    # Risk description
    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    # Likelihood inputs
    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    # Impact inputs
    with gr.Row():
        safety_dd = gr.Dropdown(IMPACT, label="Safety Impact")
        availability_dd = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality_dd = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity_dd = gr.Dropdown(IMPACT, label="Integrity Impact")

    # Run button
    run_btn = gr.Button("Run Assessment")

    # Output panels
    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    # Bind UI → pipeline
    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety_dd,
            availability_dd,
            confidentiality_dd,
            integrity_dd
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://408a786780ccf41e7a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
